# M01 — Fundamentos y entorno

[← Anterior](../M00-entorno-notebooks/02-lab-primer-notebook.ipynb) · [Siguiente →](02-lab-sesion-spark.ipynb)

Si vienes de Pandas, este notebook es el puente. Primero miramos un dataset **en Pandas** (cálculo inmediato, índice, RAM). Después el **mismo** dataset en PySpark (plan, acciones, sin índice).

Después creas el lab en `notebooks/trabajo/`. Guion: `02-lab-sesion-spark.ipynb`.

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

Ejecuta estas dos celdas. Localizan el repo y dejan una `SparkSession` lista.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark('novashop-clase-m01')
print(spark.version, spark.sparkContext.master)


## Un dataset pequeño en Pandas

Cinco pedidos inventados. En Pandas cada línea **ya calcula**. Hay índice `0..4`. Todo cabe en la RAM de este proceso Python.


In [ ]:
import pandas as pd

pedidos_pd = pd.DataFrame(
    [
        {"order_id": "O1", "status": "paid", "amount": 10.0},
        {"order_id": "O2", "status": "cancelled", "amount": 20.0},
        {"order_id": "O3", "status": "paid", "amount": 5.0},
        {"order_id": "O4", "status": "paid", "amount": 15.0},
        {"order_id": "O5", "status": "pending", "amount": 8.0},
    ]
)
pedidos_pd


`pedidos_pd` **es** la tabla. Si la imprimes, ves filas. El índice (columna de la izquierda) es de Pandas: Spark no tiene eso.

Mira tipos y forma. `dtypes` y `shape` no lanzan ningún “job”: los datos ya están en memoria.


In [ ]:
print("shape", pedidos_pd.shape)
print(pedidos_pd.dtypes)
print("índice:", list(pedidos_pd.index))
pedidos_pd.describe()


## Filtrar y agregar en Pandas

`pedidos_pd[condición]` **devuelve otro DataFrame ya calculado**. `len(...)` y `.sum()` son números ahora mismo.


In [ ]:
paid_pd = pedidos_pd[pedidos_pd["status"] == "paid"]
print("tipo de paid_pd:", type(paid_pd))
print("len (filas paid):", len(paid_pd))
print("suma amount paid:", paid_pd["amount"].sum())
paid_pd


Una columna nueva se asigna y **ya está**. `groupby` aplasta filas y te deja una tabla pequeña, también inmediata.


In [ ]:
pedidos_pd = pedidos_pd.copy()
pedidos_pd["channel"] = "web"
print(pedidos_pd[["order_id", "channel"]])

pedidos_pd.groupby("status")["amount"].agg(["count", "sum"])


## El mismo dataset en PySpark

Mismas 5 filas. `createDataFrame` no “guarda un Excel”: guarda un **plan** que sabe cómo construir esas filas. Hasta que no lances `show` / `count`, no hay tabla en pantalla.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, lit, sum as fsum

pedidos_sp = spark.createDataFrame(
    [
        Row(order_id="O1", status="paid", amount=10.0),
        Row(order_id="O2", status="cancelled", amount=20.0),
        Row(order_id="O3", status="paid", amount=5.0),
        Row(order_id="O4", status="paid", amount=15.0),
        Row(order_id="O5", status="pending", amount=8.0),
    ]
)
print("tipo:", type(pedidos_sp))
print("imprimir el objeto NO es la tabla:")
print(pedidos_sp)
pedidos_sp.printSchema()
pedidos_sp.show()


## Dónde se parecen y dónde no

| Qué haces | Pandas | PySpark |
|-----------|--------|---------|
| Ver las filas | el propio `df` / `head()` | `show()` (acción) |
| Número de filas | `len(df)` / `df.shape[0]` | `count()` (acción) |
| Tipos | `dtypes` (inferidos al crear) | `printSchema()` |
| Índice 0,1,2… | sí | **no** |
| Filtrar | `df[df.col == x]` (ya calculado) | `filter` / `where` (plan) |
| Columna nueva | `df["c"] = ...` | `withColumn` (plan) |
| Agrupar | `groupby` (ya calculado) | `groupBy` + `agg` (plan hasta `show`) |
| Dónde viven | RAM del proceso | particiones (aquí: cores del Codespace) |

Ejecuta el filtro Spark y fíjate: el `print` del objeto **no** es una tabla de 3 filas.


In [ ]:
paid_sp = pedidos_sp.filter(col("status") == "paid")
print("después del filter, ¿es una tabla?")
print(paid_sp)
print("count (ahora sí calcula):", paid_sp.count())
paid_sp.show()


Misma agregación que en Pandas: recuento y suma por `status`. Sin `show`/`collect` no ves el resultado.


In [ ]:
(
    pedidos_sp.groupBy("status")
    .agg(
        fsum("amount").alias("amount_sum"),
    )
    .show()
)


Columna nueva: en Spark **no** haces `df["channel"] = "web"` (eso es Pandas). Encadenas `withColumn` y reasignas.


In [ ]:
pedidos_sp = pedidos_sp.withColumn("channel", lit("web"))
pedidos_sp.select("order_id", "channel").show()


## El puente (y la trampa)

`toPandas()` trae **todas** las filas al driver. En 5 pedidos no pasa nada. En el fact de NovaShop (miles de líneas, y en un cluster millones) te comes la RAM.

Úsalo solo con `limit(...)` para mirar.


In [ ]:
muestra = pedidos_sp.limit(3).toPandas()
print(type(muestra))
muestra


## Transformación frente a acción (resumen)

- **Transformación** (`filter`, `withColumn`, `groupBy`, `select`): alarga el plan. No hay job en Spark UI.
- **Acción** (`show`, `count`, `collect`, `write`, `toPandas`): ejecuta. Aparece un job (puerto **4040**).

Spark no “guarda” el DataFrame como un Excel. Guarda un **plan**. Hasta una acción, la cocina está apagada.


**Siguiente:** abre el [lab](02-lab-sesion-spark.ipynb) y **crea tu** notebook.
